# ЛР 01.3 — Сравнение моделей до/после отбора признаков (TODO)

Ориентир по времени: 60 минут (+ опциональное расширение).

## Цель
Сравнить качество и скорость классических моделей на полном наборе признаков и на наборах, полученных после отбора.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    evaluate_binary_model,
    metrics_to_long_rows,
    build_segment_error_table,
    compute_threshold_metrics,
    get_binary_score_vector,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Загрузка candidate feature set
Наборы берутся из `feature_sets_wrapper_embedded.json` (Notebook 2).

In [ ]:
feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'
if feature_sets_path.exists():
    with open(feature_sets_path, 'r', encoding='utf-8') as f:
        feature_sets = json.load(f)
else:
    feature_sets = {}

print('feature sets loaded:', bool(feature_sets))

## Сравнение моделей
Обязательные модели:
- `LogisticRegression`
- `RandomForestClassifier`
- `LinearSVC`

Опциональный блок:
- `MLPClassifier` (один скрытый слой)

TODO: активируйте `RUN_MLP_BLOCK = True` и сравните с классическими моделями.

In [ ]:
RUN_MLP_BLOCK = False

model_results_rows = []

for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    available_sets = {'full': feature_names.tolist()}

    from_json = feature_sets.get(dataset_name, {})
    for set_name, feats in from_json.items():
        feats_filtered = [f for f in feats if f in set(feature_names)]
        if len(feats_filtered) >= 2:
            available_sets[set_name] = feats_filtered

    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=2500,
            class_weight='balanced',
            random_state=SEED,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=350,
            random_state=SEED,
            class_weight='balanced_subsample',
            n_jobs=-1,
        ),
        'LinearSVC': LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=SEED,
        ),
    }

    if RUN_MLP_BLOCK:
        models['MLPClassifier'] = MLPClassifier(
            hidden_layer_sizes=(32,),
            max_iter=400,
            random_state=SEED,
            early_stopping=True,
        )

    for feature_set_name, selected_features in available_sets.items():
        idx = [int(np.where(feature_names == f)[0][0]) for f in selected_features]
        X_train_sel = X_train_t[:, idx]
        X_test_sel = X_test_t[:, idx]

        for model_name, model in models.items():
            metrics = evaluate_binary_model(model, X_train_sel, y_train, X_test_sel, y_test)
            model_results_rows.extend(
                metrics_to_long_rows(
                    dataset_name=dataset_name,
                    feature_set=feature_set_name,
                    model_name=model_name,
                    metrics=metrics,
                )
            )

model_results = pd.DataFrame(model_results_rows)
model_results.head(15)

## Итоги и экспорт `model_results`

In [ ]:
model_results_path = OUTPUT_DIR / 'model_results.csv'
model_results.to_csv(model_results_path, index=False)
print(f'Saved: {model_results_path}')

summary = (
    model_results
    .pivot_table(
        index=['dataset', 'feature_set', 'model'],
        columns='metric',
        values='value',
        aggfunc='mean',
    )
    .reset_index()
    .sort_values(['dataset', 'roc_auc', 'f1', 'accuracy'], ascending=[True, False, False, False])
)
summary.head(20)

## Контрольные точки
1. Убедитесь, что есть минимум метрики `accuracy`, `f1`, `roc_auc`.
2. Проверьте, что в `model_results` присутствуют оба датасета.
3. Сравните хотя бы `full` и один отобранный набор признаков.

In [ ]:
required_columns = {'dataset', 'feature_set', 'model', 'metric', 'value', 'fit_time_sec'}
assert required_columns.issubset(model_results.columns)

assert set(model_results['dataset']) == {'medical', 'finance'}
assert {'accuracy', 'f1', 'roc_auc'}.issubset(set(model_results['metric']))

for dataset_name in ['medical', 'finance']:
    subset = model_results[model_results['dataset'] == dataset_name]
    assert 'full' in set(subset['feature_set'])

print('Проверки пройдены успешно.')

## Типичные ошибки
- Несогласованные наборы признаков между train и test.
- Сравнение моделей только по одной метрике без учета времени обучения.
- Включение MLP без фиксации random_state и без проверки сходимости.

## Финальные выводы (заполните)
TODO:
1. Какой feature set оказался лучшим для `medical` и `finance`?
2. Какая модель наиболее устойчива к сокращению признаков?
3. В каких условиях вы бы включили MLP-блок в обязательную часть курса?

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
Эти блоки требуют самостоятельной реализации и остановят ноутбук до заполнения.

### Подготовка контекста для самостоятельных заданий (`experiment_cache`)

**Контракт подготовки**

Вы создаете `experiment_cache`, где для каждого датасета хранится:
- выбранная лучшая пара `model + feature_set` (по `summary`);
- `y_true`, `y_score`, `y_pred_default` для test-части;
- `raw_test` для сегментного анализа;
- `X_full_t`, `y_full`, `feature_names`, `selected_features` для CV.

Эта ячейка должна выполняться полностью и не содержать `NotImplementedError`.

In [ ]:
from sklearn.base import clone

if 'summary' not in globals():
    raise RuntimeError('Сначала выполните базовые шаги до формирования summary.')
if 'feature_sets' not in globals():
    raise RuntimeError('Сначала выполните ячейку загрузки feature_sets.')

model_factory = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=2500, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=350,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
    'MLPClassifier': lambda: MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=400,
        random_state=SEED,
        early_stopping=True,
    ),
}

segment_feature_by_dataset = {
    'medical': 'age',
    'finance': 'credit_score',
}

experiment_cache = {}

for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X_raw, y = split_xy(df)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split_stratified(X_raw, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train_raw)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train_raw, X_test_raw)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    summary_ds = summary[summary['dataset'] == dataset_name].copy()
    if summary_ds.empty:
        raise RuntimeError(f'В summary нет строк для dataset={dataset_name}.')

    best_row = summary_ds.sort_values(['roc_auc', 'f1', 'accuracy'], ascending=False).iloc[0]
    chosen_model_name = str(best_row['model'])
    chosen_feature_set_name = str(best_row['feature_set'])

    if chosen_feature_set_name == 'full':
        selected_features = feature_names.tolist()
    else:
        selected_features = [
            feat
            for feat in feature_sets.get(dataset_name, {}).get(chosen_feature_set_name, [])
            if feat in set(feature_names)
        ]
        if len(selected_features) < 2:
            selected_features = feature_names.tolist()
            chosen_feature_set_name = 'full_fallback'

    selected_idx = [int(np.where(feature_names == feat)[0][0]) for feat in selected_features]
    X_train_sel = X_train_t[:, selected_idx]
    X_test_sel = X_test_t[:, selected_idx]

    fitted_model = clone(model_factory[chosen_model_name]())
    fitted_model.fit(X_train_sel, y_train)

    y_score, score_source = get_binary_score_vector(fitted_model, X_test_sel)
    y_pred_default = (y_score >= 0.5).astype(int)

    X_full_t = to_dense(preprocessor.fit_transform(X_raw))

    experiment_cache[dataset_name] = {
        'dataset': dataset_name,
        'model_name': chosen_model_name,
        'feature_set_name': chosen_feature_set_name,
        'selected_features': selected_features,
        'feature_names': feature_names.tolist(),
        'X_full_t': X_full_t,
        'y_full': y.reset_index(drop=True).to_numpy(),
        'y_true': y_test.reset_index(drop=True).to_numpy(),
        'y_score': y_score,
        'y_pred_default': y_pred_default,
        'raw_test': X_test_raw.reset_index(drop=True),
        'segment_feature': segment_feature_by_dataset.get(dataset_name, 'age'),
        'score_source': score_source,
    }

print('experiment_cache prepared for:', list(experiment_cache.keys()))
for ds, item in experiment_cache.items():
    print(f"{ds}: model={item['model_name']}, set={item['feature_set_name']}, score_source={item['score_source']}")

### Задание 1. Тюнинг порога классификации

**Контракт задания**

Входные переменные:
- `experiment_cache`, `compute_threshold_metrics`, `threshold_grid`.

Ожидаемый выход:
- `threshold_tuning_results` с колонками:
  `dataset`, `model`, `feature_set`, `threshold`, `precision`, `recall`, `f1`.

In [ ]:
required_columns_task1 = [
    'dataset',
    'model',
    'feature_set',
    'threshold',
    'precision',
    'recall',
    'f1',
]
threshold_tuning_results = pd.DataFrame(columns=required_columns_task1)
assert set(required_columns_task1).issubset(threshold_tuning_results.columns)

threshold_grid = np.arange(0.20, 0.81, 0.05)

# TODO(обязательно):
# 1) Для каждой записи в experiment_cache переберите threshold_grid.
# 2) Рассчитайте precision/recall/f1 через compute_threshold_metrics.
# 3) Заполните threshold_tuning_results.

raise NotImplementedError(
    'Задание 1 не завершено: заполните threshold_tuning_results по required_columns_task1.'
)

### Задание 2. CV-проверка стабильности финального feature set

**Контракт задания**

Входные переменные:
- `experiment_cache`, `StratifiedKFold`, модель из `experiment_cache[dataset]['model_name']`.

Ожидаемый выход:
- `cv_stability_results` с колонками:
  `dataset`, `model`, `feature_set`, `fold`, `accuracy`, `f1`, `roc_auc`.

In [ ]:
required_columns_task2 = [
    'dataset',
    'model',
    'feature_set',
    'fold',
    'accuracy',
    'f1',
    'roc_auc',
]
cv_stability_results = pd.DataFrame(columns=required_columns_task2)
assert set(required_columns_task2).issubset(cv_stability_results.columns)

# TODO(обязательно):
# 1) Используйте StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED).
# 2) Для каждого fold обучите модель и соберите accuracy/f1/roc_auc.
# 3) Заполните cv_stability_results.

raise NotImplementedError(
    'Задание 2 не завершено: заполните cv_stability_results по required_columns_task2.'
)

### Задание 3. Сегментный анализ ошибок и экспорт

**Контракт задания**

Входные переменные:
- `experiment_cache`, `build_segment_error_table`, результаты заданий 1-2.

Ожидаемый выход:
- `error_by_segment` с колонками:
  `dataset`, `segment_feature`, `segment`, `n`, `error_rate`, `false_positive_rate`, `false_negative_rate`;
- сохраненные файлы:
  - `outputs/threshold_tuning_results.csv`
  - `outputs/cv_stability_results.csv`
  - `outputs/error_by_segment.csv`

In [ ]:
required_columns_task3 = [
    'dataset',
    'segment_feature',
    'segment',
    'n',
    'error_rate',
    'false_positive_rate',
    'false_negative_rate',
]
error_by_segment = pd.DataFrame(columns=required_columns_task3)
assert set(required_columns_task3).issubset(error_by_segment.columns)

threshold_tuning_path = OUTPUT_DIR / 'threshold_tuning_results.csv'
cv_stability_path = OUTPUT_DIR / 'cv_stability_results.csv'
error_by_segment_path = OUTPUT_DIR / 'error_by_segment.csv'

# TODO(обязательно):
# 1) Для каждого dataset в experiment_cache вызовите build_segment_error_table.
# 2) Объедините результаты в error_by_segment.
# 3) Проверьте required columns и сохраните все три CSV.

raise NotImplementedError(
    'Задание 3 не завершено: заполните error_by_segment и сохраните threshold_tuning/cv_stability/error_by_segment.'
)